## Importação de Bibliotecas

In [1]:
# Dependências
import sys
#!{sys.executable} -m pip install --disable-pip-version-check -r ../requirements.txt -q
print('Bibliotecas instaladas')

Bibliotecas instaladas


In [2]:
# Acesso aos módulos do diretório
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Manipulação dos dados 
import pandas as pd
import numpy as np
import pickle

# Visualização dos dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

# Funções customizadas
from configs.paths import *
from configs.function_basic import *

# Avisos
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Configuração
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)

print('✅ Bibliotecas carregadas com sucesso')

Diretórios carregados com sucesso
Funções básicas carregadas com sucesso
✅ Bibliotecas carregadas com sucesso


In [3]:
# define a coluna alvo do modelo
TARGET_COL = 'FPD'

# garante reprodutibilidade dos experimentos
RANDOM_STATE = 42

# percentual máximo de valores ausentes permitido para manter a variável
PERCENTUAL_MAX_FALTANTES  = 70

# define a estratégia de escala para variáveis numéricas
ESCALA_NUMERICA = "padronizacao" # "padronizacao" → StandardScaler, "normalizacao" → MinMaxScaler

# limite de categorias distintas para decidir o tipo de encoding
LIMIAR_CARDINALIDADE = 10 # <= LIMIAR_CARDINALIDADE → One-Hot, > LIMIAR_CARDINALIDADE → Label / Ordinal

## Carregamento dos Dados

In [4]:
# Carregar dados brutos
abt00 = pd.read_parquet(RAW_DIR / 'base_tabelao.parquet')

print(f'✅ Dados carregados: {abt00.shape[0]:,} linhas e {abt00.shape[1]:,} colunas')
print(f'\nTarget (FPD) distribuição:')
print((abt00['FPD'].value_counts(normalize=True) * 100).round(2))

✅ Dados carregados: 1,272,095 linhas e 108 colunas

Target (FPD) distribuição:
FPD
0    76.65
1    23.35
Name: proportion, dtype: Float64


In [5]:
# Nome do arquivo
ARTIFACT_NAME_INFO = 'metadata.csv'

# Carregar os dados 
metadata = pd.read_csv( ARTIFACT_DIR / ARTIFACT_NAME_INFO)

## Divisão Treino/Teste (Temporal por SAFRA)

In [6]:
# Verificar SAFRAs disponíveis
if 'SAFRA' in abt00.columns:
    print(f'\n📅 SAFRAs disponíveis:')
    print(abt00['SAFRA'].value_counts().sort_index())


📅 SAFRAs disponíveis:
SAFRA
202410    203828
202411    226119
202412    225760
202501    217590
202502    198069
202503    200729
Name: count, dtype: int64


### Separação dos dados para validação temporal (Out-of-Time)

A separação dos dados é realizada com base na **safra**, respeitando a ordem temporal das observações.  
Essa abordagem, conhecida como **validação Out-of-Time (OOT)**, evita vazamento de informação e simula o comportamento real do modelo em dados futuros.

In [7]:
# garante SAFRA como inteiro
abt00['SAFRA'] = abt00['SAFRA'].astype(int)
safra_counts = abt00['SAFRA'].value_counts().sort_index()

# Definir SAFRAs de teste (Fevereiro e Março 2025)
test_safras = [202502, 202503]

# Criar máscaras
test_mask = abt00['SAFRA'].isin(test_safras)
train_mask = ~test_mask

# Separar dados
df_train = abt00[train_mask].copy()
df_test = abt00[test_mask].copy()

## Backup e Análise Inicial

In [8]:
# Backup dos dados originais
abt01 = df_train.copy()

print(f'Shape original: {abt01.shape}')

Shape original: (873297, 108)


## Tratamento para modelagem

In [9]:
# ajuste de tipagem das variáveis CEP e data de nascimento
abt01['CEP_3_digitos'] = abt01['CEP_3_digitos'].astype('Int64')
abt01['DATADENASCIMENTO'] = pd.to_datetime(abt01['DATADENASCIMENTO'], format='%d/%m/%Y', errors='coerce')

## Feature Engineering

In [10]:
# calcula idade do cliente na safra
abt01['DATADENASCIMENTO'] = pd.to_datetime(abt01['DATADENASCIMENTO'], dayfirst=True)
abt01['SAFRA_DT'] = pd.to_datetime(abt01['SAFRA'].astype(str) + '01', format='%Y%m%d')
abt01['IDADE'] = ((abt01['SAFRA_DT'] - abt01['DATADENASCIMENTO']).dt.days // 365)

# Média dos scores
abt01['SCORE_MEDIO'] = (abt01['SCORE_01'] + abt01['SCORE_02']) / 2
abt01['SCORE_DIFF'] = abt01['SCORE_01'] - abt01['SCORE_02']

In [11]:
# remove variáveis que causam vazamento, identificação ou controle temporal
ignore_cols = ['SAFRA', 'FPD', 'FPD_bureau', 'FPD_telco', 'flag_mig2', 'flag_mig2_bureau', 'flag_mig2_telco', 'NUM_CPF', 'DATADENASCIMENTO', 'SAFRA_DT']

abt01 = abt01.drop(columns=ignore_cols, errors='ignore')

>As variáveis removidas incluem identificadores únicos, variáveis de controle temporal e indicadores diretamente relacionados ao evento de inadimplência, prevenindo vazamento de informação >e garantindo aderência ao cenário real de decisão de crédito.

## Remoção de Colunas Desnecessárias

In [12]:
# filtra variáveis com muitos nulos OU cardinalidade igual a 1
df_drop = metadata[(metadata['Nulos_%'] >= PERCENTUAL_MAX_FALTANTES) | (metadata['cardinalidade'] == 1)]
var_drop = list(df_drop.variavel.values)

# efeito real do drop
qtd_excluir = abt01.columns.isin(var_drop).sum()
print(qtd_excluir)

# remover variáveis do ABT ignorando colunas inexistentes
abt01 = abt01.drop(columns=var_drop, errors='ignore')

print(var_drop)

23
['FLAG_INSTALACAO', 'PROD', 'flag_mig2_bureau', 'flag_mig2_telco', 'FLAG_INSTALACAO_cadastrais', 'flag_mig2', 'var_02', 'var_06', 'var_07', 'var_08', 'var_10', 'var_11', 'var_12', 'var_13', 'var_14', 'var_15', 'var_16', 'var_17', 'var_18', 'var_19', 'var_20', 'var_21', 'var_22', 'var_23', 'var_24', 'var_25']


In [13]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'var_drop.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(var_drop, f)

In [14]:
# Valida persistência
# recarregar do pickle e comparar
with open(artifact_path, 'rb') as f:
    var_drop_reload = pickle.load(f)

set(var_drop) == set(var_drop_reload)

True

## Tratamento de Valores Faltantes

In [15]:
# Análise de missing values restantes
abt01, stats = custom_fillna(abt01, strategy='median')

In [16]:
# Salvar a lista em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'stats_nulo.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(stats, f)

In [17]:
# Valida persistência
# reload e comparação
with open(artifact_path, 'rb') as f:
    stats_reload = pickle.load(f)

stats == stats_reload

True

## Cardinalidade

In [18]:
# colunas realmente existentes no dataframe
cols_df = set(abt01.columns)

# selecionar variáveis categóricas com baixa cardinalidade para One-Hot Encoding
df_categ_onehot = metadata[(metadata['cardinalidade'] <= LIMIAR_CARDINALIDADE) & (metadata['tipo'] == 'categorica') & (metadata['variavel'].isin(cols_df))]

# selecionar variáveis categóricas com alta cardinalidade para Label Encoding
df_categ_labelenc = metadata[(metadata['cardinalidade'] > LIMIAR_CARDINALIDADE) & (metadata['tipo'] == 'categorica') & (metadata['variavel'].isin(cols_df))]

# extraindo lista das variáveis para Encoding
lista_onehot = list(df_categ_onehot.variavel.values)
lista_lenc = list(df_categ_labelenc.variavel.values)

print('Lista de vars para Label Encoding: ',lista_lenc)
print('Lista de vars para OneHot Encoding: ',lista_onehot)

Lista de vars para Label Encoding:  ['CEP_3_digitos']
Lista de vars para OneHot Encoding:  ['STATUSRF']


### OneHotEncoder (Baixa cardinalidade)

In [19]:
# Instanciando o encoder
encoder_onehot = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

# Aplicando o one-hot encoding
encoded_onehot = encoder_onehot.fit_transform(abt01[lista_onehot])

# Criando um DataFrame com as colunas codificadas, mantendo o índice original
encoded_cols = encoder_onehot.get_feature_names_out(lista_onehot)
encoded_df = pd.DataFrame(encoded_onehot, columns=encoded_cols, index=abt01.index)

# Concatenando o DataFrame codificado com o DataFrame original
abt01  = pd.concat([abt01.drop(lista_onehot, axis=1), encoded_df], axis=1)

# Salva o encoder e a lista de colunas em um arquivo .pkl
data_onehot = {
    'encoder': encoder_onehot,
    'input_columns': lista_onehot,
    'output_columns': list(encoded_cols)
}

### LabelEncoder (Alta cardinalidade)

In [20]:
# Aplicando LabelEncoder nas colunas desejadas
encoders_lenc = {}

for col in lista_lenc:
    encoder_lenc = LabelEncoder()
    abt01[col] = encoder_lenc.fit_transform(abt01[col])
    # Armazena o encoder para a coluna atual em um dicionário
    encoders_lenc[col] = encoder_lenc

# Salva o dicionário de encoders e a lista de colunas em um arquivo .pkl
data_lenc = {
    'encoders': encoders_lenc,
    'columns': lista_lenc
}

In [21]:
# Salvar a lista em um arquivo .pkl (LabelEncoder)
artifact_path_lenc = Path(ARTIFACT_DIR) / 'data_lenc.pkl'

with open(artifact_path_lenc, 'wb') as f:
    pickle.dump(data_lenc, f)

# Salvar a lista em um arquivo .pkl (OneHotEncoder)
artifact_path_onehot = Path(ARTIFACT_DIR) / 'data_onehot.pkl'

with open(artifact_path_onehot, 'wb') as f:
    pickle.dump(data_onehot, f)

In [23]:
# Validar persistência dos encoders (Label + OneHot)

# reload LabelEncoder
with open(artifact_path_lenc, 'rb') as f:
    data_lenc_reload = pickle.load(f)

# reload OneHotEncoder
with open(artifact_path_onehot, 'rb') as f:
    data_onehot_reload = pickle.load(f)

# 1️⃣ Validação estrutural
assert data_lenc_reload.keys() == data_lenc.keys()
assert data_onehot_reload.keys() == data_onehot.keys()

# 2️⃣ Validação de colunas
assert data_lenc_reload['columns'] == data_lenc['columns']
assert data_onehot_reload['input_columns'] == data_onehot['input_columns']
assert data_onehot_reload['output_columns'] == data_onehot['output_columns']

# 3️⃣ Validação funcional (o que importa de verdade)
for col, enc in data_lenc['encoders'].items():
    enc_reload = data_lenc_reload['encoders'][col]
    assert np.array_equal(enc.classes_, enc_reload.classes_)

assert np.array_equal(
    data_onehot['encoder'].categories_,
    data_onehot_reload['encoder'].categories_
)

print('✅ Encoders salvos, recarregados e validados com sucesso')


✅ Encoders salvos, recarregados e validados com sucesso


## Preparando os dados de Teste

In [24]:
# função para padronizar tratamento inicial e engenharia básica de features
def preprocess_base(df):
    df = df.copy()

    # tipagem
    df['CEP_3_digitos'] = df['CEP_3_digitos'].astype('Int64')
    df['DATADENASCIMENTO'] = pd.to_datetime(df['DATADENASCIMENTO'], format='%d/%m/%Y', errors='coerce')

    # datas auxiliares
    df['SAFRA_DT'] = pd.to_datetime(df['SAFRA'].astype(str) + '01', format='%Y%m%d', errors='coerce')

    # idade na safra
    df['IDADE'] = ((df['SAFRA_DT'] - df['DATADENASCIMENTO']).dt.days // 365)

    # features de score
    df['SCORE_MEDIO'] = (df['SCORE_01'] + df['SCORE_02']) / 2
    df['SCORE_DIFF'] = df['SCORE_01'] - df['SCORE_02']

    # colunas a remover
    ignore_cols = ['SAFRA', 'FPD', 'FPD_bureau', 'FPD_telco', 'flag_mig2', 'flag_mig2_bureau', 'flag_mig2_telco', 'NUM_CPF', 'DATADENASCIMENTO', 'SAFRA_DT']

    df = df.drop(columns=ignore_cols, errors='ignore')

    return df

In [25]:
# Backup dos dados originais
abt01_test = df_test.copy()

abt01_test  = preprocess_base(abt01_test)

In [26]:
# carregar colunas a remover
with open(Path(ARTIFACT_DIR) / 'var_drop.pkl', 'rb') as f:
    var_drop = pickle.load(f)

# aplicar no conjunto de teste
abt01_test = abt01_test.drop(columns=var_drop, errors='ignore')

In [27]:
# carregar estatísticas de imputação
with open(Path(ARTIFACT_DIR) / 'stats_nulo.pkl', 'rb') as f:
    stats = pickle.load(f)

# numéricas
for col, value in stats['numerical'].items():
    if col in abt01_test.columns:
        abt01_test[col] = abt01_test[col].fillna(value)

# categóricas
cat_cols = abt01_test.select_dtypes(include=['object']).columns
abt01_test[cat_cols] = abt01_test[cat_cols].fillna(stats['categorical_fill'])


In [28]:
# carregar encoders e aplicar encoding na base de teste
artifact_path_lenc = Path(ARTIFACT_DIR) / 'data_lenc.pkl'
artifact_path_onehot = Path(ARTIFACT_DIR) / 'data_onehot.pkl'

# load artefatos
with open(artifact_path_lenc, 'rb') as f:
    data_lenc = pickle.load(f)

with open(artifact_path_onehot, 'rb') as f:
    data_onehot = pickle.load(f)

# ---------------- LabelEncoder ----------------
encoders = data_lenc["encoders"]
columns_lenc = data_lenc["columns"]

for col in columns_lenc:
    if col in abt01_test.columns:
        mapping = {label: idx for idx, label in enumerate(encoders[col].classes_)}
        abt01_test[col] = abt01_test[col].map(mapping).fillna(-1).astype(int)

# ---------------- OneHotEncoder ----------------
onehot_encoder = data_onehot["encoder"]
onehot_cols = data_onehot["input_columns"]
output_cols = data_onehot["output_columns"]

X_onehot = onehot_encoder.transform(abt01_test[onehot_cols])

df_onehot = pd.DataFrame(
    X_onehot,
    columns=output_cols,
    index=abt01_test.index
)

# remove colunas originais e concatena onehot
abt01_test = pd.concat(
    [abt01_test.drop(onehot_cols, axis=1), df_onehot],
    axis=1
)

## Sanity check

In [29]:
cols_train = set(abt01.columns)
cols_test = set(abt01_test.columns)

print('Só no treino:', cols_train - cols_test)
print('Só no teste:', cols_test - cols_train)

Só no treino: set()
Só no teste: set()


In [30]:
# Análise de missing values
analyze_missing_values(abt01_test, plot=False)

✅ Excellent! No missing values found


,Variable,Missing_Values,Percentage


# Trazer o target para a tabela pós dataprep

In [31]:
# Checar se número de linhas bate
print("Treino:")
print("linhas abt01:", len(abt01))
print("linhas df_train:", len(df_train))

print("\nTeste:")
print("linhas abt01_test:", len(abt01_test))
print("linhas df_test:", len(df_test))

Treino:
linhas abt01: 873297
linhas df_train: 873297

Teste:
linhas abt01_test: 398798
linhas df_test: 398798


In [32]:
# Reanexar target antes de salvar

# Para treino
abt01_train_final = abt01.copy()
abt01_train_final['FPD'] = df_train['FPD']  # reanexa target original

# Para teste
abt01_test_final = abt01_test.copy()
abt01_test_final['FPD'] = df_test['FPD']    # reanexa target original

## Salvamento dos Dados Processados

In [33]:
# Salvar datasets processados e lista de features

print('\n💾 Salvando dados processados...')

# Dataset completo (opcional)
abt01_train_final.to_parquet(PROCESSED_DIR / 'abt01_train.parquet', index=False)
abt01_test_final.to_parquet(PROCESSED_DIR / 'abt01_test.parquet', index=False)
print(f'   ✓ Treino salvo: {PROCESSED_DIR / "abt01_train.parquet"}')
print(f'   ✓ Teste salvo: {PROCESSED_DIR / "abt01_test.parquet"}')

# Salvar lista de features (excluindo a target)
features_list = [c for c in abt01.columns if c != 'FPD']
with open(ARTIFACT_DIR / 'features_list.pkl', 'wb') as f:
    pickle.dump(features_list, f)
print(f'   ✓ Lista de features salva em: {ARTIFACT_DIR / "features_list.pkl"}')

print(f'\n✅ Todos os dados processados foram salvos')


💾 Salvando dados processados...
   ✓ Treino salvo: C:\Users\billy.reis\Desktop\hackathon_pod\hackathon\data\processed\abt01_train.parquet
   ✓ Teste salvo: C:\Users\billy.reis\Desktop\hackathon_pod\hackathon\data\processed\abt01_test.parquet
   ✓ Lista de features salva em: C:\Users\billy.reis\Desktop\hackathon_pod\hackathon\artifact\features_list.pkl

✅ Todos os dados processados foram salvos


## Resumo do Tratamento

In [ ]:
# Resumo do tratamento dos dados

print('\n' + '='*60)
print('RESUMO DO TRATAMENTO DOS DADOS')
print('='*60)

# Shapes - comparando treino original vs final processado
print(f'\n✅ Shape original (treino antes do processamento): {df_train.shape[0]:,} registros × {df_train.shape[1]} colunas')
print(f'✅ Shape final (treino após processamento): {abt01_train_final.shape[0]:,} registros × {abt01_train_final.shape[1]} colunas')

# Colunas removidas e novas features
cols_original = df_train.shape[1]
cols_final = abt01_train_final.shape[1]
new_features = 3  # IDADE, SCORE_MEDIO, SCORE_DIFF
cols_removed = cols_original + new_features - cols_final

print(f'\n✅ Colunas removidas no pipeline: {cols_removed}')
print(f'✅ Novas features criadas: {new_features} (IDADE, SCORE_MEDIO, SCORE_DIFF)')
print(f'✅ Variação líquida: {cols_final - cols_original:+d} colunas')

# Missing values - comparando original vs final
missing_orig = df_train.isnull().sum().sum()
missing_final = abt01_train_final.isnull().sum().sum()
print(f'\n✅ Missing values originais: {missing_orig:,}')
print(f'✅ Missing values finais: {missing_final:,}')
print(f'✅ Redução de missing: {missing_orig - missing_final:,} ({((missing_orig - missing_final) / max(missing_orig, 1) * 100):.1f}%)')

# Divisão temporal por SAFRA
print(f'\n✅ Divisão treino/teste: TEMPORAL por SAFRA')
treino_safras = sorted(df_train["SAFRA"].unique().tolist())
teste_safras = sorted(df_test["SAFRA"].unique().tolist())
print(f'   📊 Treino: SAFRAs {treino_safras} → {abt01_train_final.shape[0]:,} registros')
print(f'   📊 Teste: SAFRAs {teste_safras} → {abt01_test_final.shape[0]:,} registros')

# Distribuição da variável target
print(f'\n✅ Distribuição do target (FPD) no treino:')
fpd_dist = abt01_train_final['FPD'].value_counts(normalize=True).sort_index()
for label, pct in fpd_dist.items():
    print(f'   • Classe {label}: {pct*100:.2f}%')

print(f'\n✅ Dados prontos para modelagem!')
print('='*60)

## Nomarlização

In [ ]:
# Escala numérica: normalização ou padronização =====
if ESCALA_NUMERICA == "padronizacao":
    scaler_num = StandardScaler()
elif ESCALA_NUMERICA == "normalizacao":
    scaler_num = MinMaxScaler()
else:
    raise ValueError("ESCALA_NUMERICA deve ser 'padronizacao' ou 'normalizacao'")

# Instanciando o scaler
scaler = scaler_num

# Selecionando colunas numéricas
numeric_cols = abt01.select_dtypes(include=['float64', 'int64','int32']).columns

# Aplicando a normalização
abt01[numeric_cols] = scaler.fit_transform(abt01[numeric_cols])

In [ ]:
# Salva o scaler em um arquivo .pkl
artifact_path = Path(ARTIFACT_DIR) / 'scaler.pkl'

with open(artifact_path, 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
abt01.head()